# Calculating Power-law scaling in fluctuations of information

In [6]:
import gc
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [7]:
CORPUS_NAME = 'null'

In [15]:
DATA_PATH = '../data/null-avar'
IS_REAL_PATH = '../data/ckpts/rosen'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [9]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [29]:
fs = [f for f in grab_all_files(DATA_PATH) if (not f.split('/')[-1].startswith('.'))]
len(fs)

1669

##### Some last checks and corrections

In [ ]:
IS_REAL_DATA = grab_all_files(IS_REAL_PATH)

In [25]:
f = open('/Users/zacharyrosen/Desktop/file-list.txt', 'w')
f.write('\n'.join([fi.split('/')[-1] for fi in fs if ('CANDOR-' in fi)]))
f.close()

In [26]:
set([fi.split('/')[-1] for fi in fs]).difference(set([fi.split('/')[-1] for fi in IS_REAL_DATA]))

{'callfriend-eng_s-callfriend-eng_n-callhome-ORIG.parquet'}

In [28]:
keep = os.path.join(DATA_PATH, 'callfriend-eng_s-callfriend-eng_n-callhome-ORIG.parquet')
os.remove(keep.replace('-ORIG', ''))
os.rename(keep, keep.replace('-ORIG', ''))

## Processing individual datasets

In [30]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [31]:
COEF_VAR = 'AVAR'

In [32]:
param_list = 'time_delta'

In [33]:
df_scale, df_regression, k_docs, pct = [], [], dict(), 1.

In [34]:
for f in tqdm(fs):

    # print(f.split('/')[-1])

    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()


    df['self'] = (df['x_speaker'] == df['y_speaker'])
    df['x_turn_id_'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df[param_list] = (df['y_turn_id'] - df['x_turn_id_']) + 1

    df = df.loc[
        (~df[COEF_VAR].isna())
        & (~df[param_list].isna())
    ]

    for corpus in df['corpus'].unique():

        if CORPUS_NAME not in k_docs.keys():
            k_docs[CORPUS_NAME] = {
                True: 0,
                False: 0
            }

        sub = df.loc[df['corpus'].isin([corpus])]

        keep_data = sub['x_turn_id'].unique()

        keep_data = np.random.choice(keep_data, size=(int(np.ceil(pct*len(keep_data)),),), replace=False)

        sub = sub.loc[sub['x_turn_id'].isin(keep_data)]
        gc.collect()

        groups = sub.groupby(by=['conversation_id', 'x_speaker', 'self'])
        for labels, dfi in groups:

            if len(dfi):
                k_docs[CORPUS_NAME][labels[2]] += 1

                b,a = np.polyfit(np.log(dfi[param_list].sample(frac=1.).values), np.log(dfi['AVAR'].sample(frac=1.).values + 1e-9), 1)

                df_regression += [
                    {
                        'corpus': CORPUS_NAME,
                        'cid': labels[0],
                        'speaker': labels[1],
                        'self': labels[2],
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/1669 [00:00<?, ?it/s]

In [11]:
# df_regression.isna().sum()

In [12]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [13]:
# if not os.path.exists(os.path.join(OUTPUTS_PATH,'all-null.csv')):
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all-null.csv'),
#         index=False,
#         encoding='utf-8'
#     )
#
# else:
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all-null.csv'),
#         index=False,
#         header=False,
#         encoding='utf-8',
#         mode='a'
#     )

## Corpus level analysis of results

In [35]:
split_by = ['corpus', 'self']

In [36]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [37]:
df0['t'] = df0['b'] / df0['se']

In [38]:
df0.head(n=20)

,corpus,self,b,std,k,se,t
0,null,False,-0.045657,0.981434,12604,0.008742,-5.222784
1,null,True,-0.056821,1.259633,11290,0.011855,-4.793042


In [39]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, CORPUS_NAME+'.csv'),
    index=False, encoding='utf-8'
)